### Imports

In [1]:
import numpy as np
import os

from cgrbptools import CGRBPTopology
from cgrbptools import ConfBuilder
from cgrbptools import polycg
from cgrbptools import rescale_stiff

---
---
# Sequence-Dependent Models
---
---

### Set output name

In [2]:
base_filename = "Test/seqdep_example"

outdir = os.path.dirname(base_filename)
if outdir:
    os.makedirs(outdir, exist_ok=True)

## Set model parameters

In [3]:
num_bp = 4000
model = 'cgNAplus'
closed = True

coupling_range = 2
composite_size = 10

seq = ''.join(np.random.default_rng(5).choice(list('ACGT'), num_bp))

# set discretization length (nm per bead) based on the composite size
disc_len = 0.34 * composite_size  # nm per bead

# closed topology
closed = True

# optional FENE bond 
# Set to (K, Rc, R0) to add a native FENE bond, or None to disable.
fene_coeffs = np.array([500/disc_len**2, 1.15*disc_len, 1.30*disc_len])  # (K, Rc, R0) for the FENE bond
fene_sim_units = False # fene coefficients expressed in simulation units (True) or in nm/kT

# formatting / units
decimals    = 4      # decimal places in the written coefficients
mass        = 1.0    # mass per bead
unit_length = 3.4    # nm per length unit (1.0 -> keep nm). If equal to disc_len, the junctions are written in units of disc_len. The translational stiffness is rescaled accordingly.
unit_energy = 1.0    # kT per energy unit

# generate parameters for the given sequence and model
params = polycg.gen_params(
    model,
    seq,
    composite_size=composite_size,
    closed=closed,
    verbose=True,
)

# Select shape and stiffness matrices
if params.is_coarse_grained:
    stiff = params.cg_stiffmat
    shape = params.cg_shape_params
else: 
    stiff = params.stiffmat
    shape = params.shape_params

print(stiff.shape)
print(shape.shape)

Generating 40 stiffness blocks for 4080 base-pair steps
Progress: |████████████████████████████████████████| 100.0% Block 40/40 (bps 3900-4080) ETA 00:00 (elapsed 00:06)
Coarse-graining 40 blocks (4080 total bps, composite_size=10)
Progress: |████████████████████████████████████████| 100.0% Block 40/40 (bps 3900-4080) ETA 00:00 (elapsed 00:08)
(2400, 2400)
(400, 6)


### Stiffness rescaling

In [4]:
# rescales the stiffness for all stepwise degrees of freedom (dofs) by the given factors. 

scale = np.ones(6)
scale[5] = 0.7

dofnames = ['Tilt', 'Roll', 'Twist', 'Shift', 'Slide', 'Rise']
for i in range(len(scale)):
    if scale[i] != 1.0:
        print(f'rescaling stiffness matrix entry {i} ({dofnames[i]}) by factor {scale[i]:.3f}')
        stiff = rescale_stiff(stiff,scale[i],entries=[i])

rescaling stiffness matrix entry 5 (Rise) by factor 0.700


### Initialize Topology

In [5]:

nbps = num_bp if closed else num_bp - 1

##################################################
########## Generate topology #####################
topol = CGRBPTopology(
    coupling_range=coupling_range,
    decimals=decimals,
    check_existing_types=True,
    closed=closed
)

# Optional native FENE bond (set before parameters so it is baked into the
# bond coefficients when couplings are built).
if fene_coeffs is not None:
    topol.set_fene(*fene_coeffs, sim_units=fene_sim_units)

topol.set_params(shape,stiff)
topol.set_unit_energy(unit_energy)
topol.set_unit_length(unit_length)

topol.set_sequence(seq,chars_per_atom=composite_size,centered=False)

Initializing couplings...
Initializing couplings...


### Write Database file

In [6]:
##################################################
########## Write topology database ###############
topol.write_database(base_filename, add_extension=True)

# Build Configuration

### Chose Type

In [7]:
# options: 'circular', 'straight', 'gs', 'tresfoil'  
conf_type = 'circ'
conf_type = 'straight'
conf_type = 'tresfoil'

box_extension = 0.05
square_box = True
excess_link = 20.0

### Built-in configuration type

In [8]:
if conf_type in ConfBuilder.mapping.keys():
    conf = ConfBuilder.build(
        topol,
        conf_type,
        mass=mass,
        excess_link=excess_link,
    )
    box = conf.extended_bounds(box_extension, square_box=square_box)

    conf.write_datafile(
        base_filename,
        topol,
        box=box,
        hybrid=False,
        include_coeffs=False,
        add_extension=True,
        box_decimals=decimals,
        overwrite=True,
    )
    conf.visualize_xyz(base_filename)

    if conf_type in ['circ', 'circular'] and not closed:
        print("Warning: circular configuration requested for an open topology. The ends will be connected in the configuration.")

### Parse custom configuration via spline interpolation of tracepoints

### Tresfoil config

In [9]:
def trefoil_tracepoints(length: float, spacing: float) -> np.ndarray:
    """
    Tracepoints along a trefoil knot, equally spaced along its contour.

    The knot is the curve (sin t + 2 sin 2t, cos t - 2 cos 2t, -sin 3t), t in [0, 2 pi), scaled to
    the contour length length. Since the curve is closed, the contour is divided into
    N = round(length / spacing) equal parts, i.e. the actual spacing is length / N. The first
    tracepoint is not repeated at the end.

    Parameters
    ----------
    length : float
        Total contour length of the knot.
    spacing : float
        Distance between consecutive tracepoints along the contour (same unit as length).

    Returns
    -------
    np.ndarray
        Tracepoints, shape (N, 3).
    """
    if not (np.isfinite(length) and length > 0 and np.isfinite(spacing) and spacing > 0):
        raise ValueError(f'length and spacing must be positive, got {length} and {spacing}.')
    num = int(round(length / spacing))
    if num < 3:
        raise ValueError(f'length / spacing gives {num} tracepoints, at least 3 are required.')

    # arc length s(t) on a fine grid (trapezoidal rule), inverted by interpolation
    t = np.linspace(0.0, 2 * np.pi, 2**16 + 1)
    speed = np.sqrt(
        (np.cos(t) + 4 * np.cos(2 * t)) ** 2
        + (-np.sin(t) + 4 * np.sin(2 * t)) ** 2
        + (3 * np.cos(3 * t)) ** 2
    )
    s = np.concatenate([[0.0], np.cumsum(0.5 * (speed[1:] + speed[:-1]) * np.diff(t))])
    tk = np.interp(np.arange(num) * s[-1] / num, s, t)

    points = np.column_stack([
        np.sin(tk) + 2 * np.sin(2 * tk),
        np.cos(tk) - 2 * np.cos(2 * tk),
        -np.sin(3 * tk),
    ])
    return points * (length / s[-1])


In [10]:
if conf_type == 'tresfoil':
    points = trefoil_tracepoints(num_bp*0.34, 0.34*composite_size)
    conf = ConfBuilder.from_tracepoints(
        topol,
        points,
        excess_link=excess_link,
        mass=mass,
        rescale=True,
    )

    box = conf.extended_bounds(box_extension, square_box=square_box)
    
    conf.write_datafile(
        base_filename,
        topol,
        box=box,
        hybrid=False,
        include_coeffs=False,
        add_extension=True,
        box_decimals=decimals,
        overwrite=True,
    )
    conf.visualize_xyz(base_filename)

    if not closed:
        print("Warning: trefoil configuration requested for an open topology. The ends will be connected in the configuration.")

### Example for importing externally generated poses

In [11]:
if conf_type == 'tresfoil':

    # generate poses with the same routine as before
    points = trefoil_tracepoints(num_bp*0.34, 0.34*composite_size)
    conf = ConfBuilder.from_tracepoints(
        topol,
        points,
        excess_link=excess_link,
        mass=mass,
        rescale=True,
    )
    # export poses
    poses = conf.poses.copy()

    # import poses into ConfBuilder
    conf = ConfBuilder.from_poses(
        poses,
        topology=topol,
        mass=mass
    )

    box = conf.extended_bounds(box_extension, square_box=square_box)
    conf.write_datafile(
        base_filename,
        topol,
        box=box,
        hybrid=False,
        include_coeffs=False,
        add_extension=True,
        box_decimals=decimals,
        overwrite=True,
    )
    conf.visualize_xyz(base_filename)

### ChimeraX visualization

In [ ]:
conf.visualize_chimerax(base_filename, include_bps_triads=True)